In [7]:
# %% [markdown]
# # Bike-sharing playground
#
# **Карта блоков (6 штук):**
# 1. Config + rng
# 2. Facilities (stations + depots → facilities)
# 3. Initial state (capacity + inventory)
# 4. Trips → demand + returns
# 5. Network (edge rules)
# 6. Simulate (demand − returns + per period → history)

# %% 1. Config + rng
import numpy as np
import pandas as pd

cfg = {
    "seed": 42,
    "n_stations": 5,
    "n_depots": 2,
    "n_periods": 2,
    "start_date": "2025-01-01",
    "trips_per_period_per_station": 8,
    "station_capacity_range": (15, 35),
    "depot_initial_stock": 50,
}

rng = np.random.default_rng(seed=cfg["seed"])

periods = pd.DataFrame({
    "period_id": [f"P{i+1}" for i in range(cfg["n_periods"])],
    "date": pd.date_range(cfg["start_date"], periods=cfg["n_periods"], freq="D"),
})

# %% 2. Facilities
n_stations, n_depots = cfg["n_stations"], cfg["n_depots"]

stations = pd.DataFrame({
    "facility_id": [f"S{i+1}" for i in range(n_stations)],
    "facility_type": "station",
    "lat": np.round(rng.uniform(40.70, 40.76, size=n_stations), 4),
    "lon": np.round(rng.uniform(-74.02, -73.95, size=n_stations), 4),
})

depots = pd.DataFrame({
    "facility_id": [f"D{i+1}" for i in range(n_depots)],
    "facility_type": "depot",
    "lat": np.round(rng.uniform(40.70, 40.76, size=n_depots), 4),
    "lon": np.round(rng.uniform(-74.02, -73.95, size=n_depots), 4),
})

facilities = pd.concat([stations, depots], ignore_index=True)

# %% 3. Initial state
low, high = cfg["station_capacity_range"]
station_capacities = rng.integers(low, high + 1, size=n_stations)

facility_capacity = pd.DataFrame({
    "facility_id": stations["facility_id"],
    "commodity_category": "manual_bike",
    "capacity": station_capacities,
})

# initial stock: 50%-100% of station capacity; depots get fixed stock
station_initial_stock = rng.integers(low=station_capacities // 2, high=station_capacities + 1)
depot_initial_stock = np.full(n_depots, cfg["depot_initial_stock"])

inventory_initial = pd.DataFrame({
    "facility_id": facilities["facility_id"],
    "commodity_category": "manual_bike",
    "quantity": np.concatenate([station_initial_stock, depot_initial_stock]),
})

# %% 4. Trips → demand + returns
station_ids = stations["facility_id"].to_numpy()
trip_rows = []
for pid in periods["period_id"]:
    n_trips = int(rng.poisson(lam=n_stations * cfg["trips_per_period_per_station"]))
    for _ in range(n_trips):
        # no self-loops: can't rent and return at same station
        start, end = rng.choice(station_ids, size=2, replace=False)
        trip_rows.append({
            "period_id": pid,
            "start_station_id": start,
            "end_station_id": end,
            "commodity_category": "manual_bike",
            "quantity": 1,
        })
trips = pd.DataFrame(trip_rows)

# demand: bikes taken from stations (rentals)
demand = (
    trips
    .groupby(["period_id", "start_station_id", "commodity_category"], as_index=False)["quantity"]
    .sum()
    .rename(columns={"start_station_id": "facility_id"})
)

# returns: bikes brought back to stations (end of ride)
returns = (
    trips
    .groupby(["period_id", "end_station_id", "commodity_category"], as_index=False)["quantity"]
    .sum()
    .rename(columns={"end_station_id": "facility_id"})
)

# %% 5. Network
edge_rules = pd.DataFrame({
    "source_type": ["station", "depot", "station"],
    "target_type": ["station", "station", "depot"],
    "modal_type": ["road", "road", "road"],
    "description": [
        "Between all stations",
        "Depot to station delivery",
        "Station to depot return",
    ],
})

# %% 6. Simulate
inventory = inventory_initial.set_index("facility_id")["quantity"].copy()
history = [{"period": "init", **inventory.to_dict()}]

for _, p in periods.iterrows():
    period_id = p["period_id"]

    # demand consumes
    period_demand = demand[demand["period_id"] == period_id].set_index("facility_id")["quantity"]
    inv_after_demand = inventory.subtract(period_demand, fill_value=0)

    # returns replenish
    period_returns = returns[returns["period_id"] == period_id].set_index("facility_id")["quantity"]
    inv_after_returns = inv_after_demand.add(period_returns, fill_value=0)

    inventory = inv_after_returns
    history.append({"period": period_id, **inventory.to_dict()})

result = pd.DataFrame(history).set_index("period").T
result.index.name = "facility_id"

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd


class BikeShareMockData:
    """Generate minimal mock data for bike-sharing simulation."""

    def __init__(self, cfg: dict):
        self._cfg = cfg
        self._rng = np.random.default_rng(seed=cfg["seed"])

        self.periods: pd.DataFrame = pd.DataFrame()
        self.facilities: pd.DataFrame = pd.DataFrame()
        self.facility_capacity: pd.DataFrame = pd.DataFrame()
        self.inventory_initial: pd.DataFrame = pd.DataFrame()
        self.trips: pd.DataFrame = pd.DataFrame()
        self.demand: pd.DataFrame = pd.DataFrame()
        self.returns: pd.DataFrame = pd.DataFrame()
        self.edge_rules: pd.DataFrame = pd.DataFrame()

    def generate(self) -> BikeShareMockData:
        """Main entry point: config → all tables on self."""
        self.periods = self._make_periods()
        stations, depots = self._make_facilities()
        self.facilities = pd.concat([stations, depots], ignore_index=True)
        self.facility_capacity, self.inventory_initial = self._make_initial_state(
            stations, self.facilities,
        )
        self.trips, self.demand, self.returns = self._make_demand_and_returns(
            stations, self.periods,
        )
        self.edge_rules = self._make_edge_rules()
        return self

    # ── periods ──

    def _make_periods(self) -> pd.DataFrame:
        n = self._cfg["n_periods"]
        return pd.DataFrame({
            "period_id": [f"P{i+1}" for i in range(n)],
            "date": pd.date_range(self._cfg["start_date"], periods=n, freq="D"),
        })

    # ── facilities ──

    def _make_facilities(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        stations = self._make_facility_group("S", "station", self._cfg["n_stations"])
        depots = self._make_facility_group("D", "depot", self._cfg["n_depots"])
        return stations, depots

    def _make_facility_group(self, prefix: str, facility_type: str, n: int) -> pd.DataFrame:
        return pd.DataFrame({
            "facility_id": [f"{prefix}{i+1}" for i in range(n)],
            "facility_type": facility_type,
            "lat": np.round(self._rng.uniform(40.70, 40.76, size=n), 4),
            "lon": np.round(self._rng.uniform(-74.02, -73.95, size=n), 4),
        })

    # ── initial state ──

    def _make_initial_state(
        self, stations: pd.DataFrame, facilities: pd.DataFrame,
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
        station_capacities = self._random_station_capacities(len(stations))
        capacity = self._make_facility_capacity(stations, station_capacities)
        inventory = self._make_inventory_initial(facilities, station_capacities)
        return capacity, inventory

    def _random_station_capacities(self, n: int) -> np.ndarray:
        low, high = self._cfg["station_capacity_range"]
        return self._rng.integers(low, high + 1, size=n)

    def _make_facility_capacity(
        self, stations: pd.DataFrame, station_capacities: np.ndarray,
    ) -> pd.DataFrame:
        return pd.DataFrame({
            "facility_id": stations["facility_id"],
            "commodity_category": "manual_bike",
            "capacity": station_capacities,
        })

    def _make_inventory_initial(
        self, facilities: pd.DataFrame, station_capacities: np.ndarray,
    ) -> pd.DataFrame:
        # stations: 50-100% of capacity; depots: fixed stock
        station_stock = self._rng.integers(
            low=station_capacities // 2, high=station_capacities + 1,
        )
        depot_stock = np.full(self._cfg["n_depots"], self._cfg["depot_initial_stock"])
        return pd.DataFrame({
            "facility_id": facilities["facility_id"],
            "commodity_category": "manual_bike",
            "quantity": np.concatenate([station_stock, depot_stock]),
        })

    # ── demand & returns ──

    def _make_demand_and_returns(
        self, stations: pd.DataFrame, periods: pd.DataFrame,
    ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        trips = self._generate_trips(stations, periods)
        demand = self._aggregate_trips(trips, "start_station_id")
        returns = self._aggregate_trips(trips, "end_station_id")
        return trips, demand, returns

    def _generate_trips(
        self, stations: pd.DataFrame, periods: pd.DataFrame,
    ) -> pd.DataFrame:
        station_ids = stations["facility_id"].to_numpy()
        lam = len(station_ids) * self._cfg["trips_per_period_per_station"]
        rows = []
        for period_id in periods["period_id"]:
            n_trips = int(self._rng.poisson(lam=lam))
            for _ in range(n_trips):
                start, end = self._rng.choice(station_ids, size=2, replace=False)
                rows.append({
                    "period_id": period_id,
                    "start_station_id": start,
                    "end_station_id": end,
                    "commodity_category": "manual_bike",
                    "quantity": 1,
                })
        return pd.DataFrame(rows)

    def _aggregate_trips(self, trips: pd.DataFrame, station_column: str) -> pd.DataFrame:
        return (
            trips
            .groupby(["period_id", station_column, "commodity_category"], as_index=False)["quantity"]
            .sum()
            .rename(columns={station_column: "facility_id"})
        )

    # ── network ──

    def _make_edge_rules(self) -> pd.DataFrame:
        return pd.DataFrame({
            "source_type": ["station", "depot", "station"],
            "target_type": ["station", "station", "depot"],
            "modal_type": ["road", "road", "road"],
            "description": [
                "Between all stations",
                "Depot to station delivery",
                "Station to depot return",
            ],
        })

In [ ]:
class BikeShareSimulator:
    """Minimal simulator: inventory evolves under demand and returns."""

    def run(
        self, inventory_initial: pd.DataFrame, demand: pd.DataFrame,
        returns: pd.DataFrame, periods: pd.DataFrame,
    ) -> pd.DataFrame:
        """Run simulation across all periods, return history table."""
        inventory = inventory_initial.set_index("facility_id")["quantity"].copy()
        history = [{"period": "init", **inventory.to_dict()}]

        for _, period in periods.iterrows():
            inventory = self._step(inventory, demand, returns, period["period_id"])
            history.append({"period": period["period_id"], **inventory.to_dict()})

        return self._build_result(history)

    def _step(
        self, inventory: pd.Series, demand: pd.DataFrame,
        returns: pd.DataFrame, period_id: str,
    ) -> pd.Series:
        """Single period: subtract demand, add returns."""
        period_demand = self._filter_period(demand, period_id)
        inv_after_demand = inventory.subtract(period_demand, fill_value=0)

        period_returns = self._filter_period(returns, period_id)
        inv_after_returns = inv_after_demand.add(period_returns, fill_value=0)

        return inv_after_returns

    def _filter_period(self, table: pd.DataFrame, period_id: str) -> pd.Series:
        """Extract quantity series for a single period."""
        return (
            table[table["period_id"] == period_id]
            .set_index("facility_id")["quantity"]
        )

    def _build_result(self, history: list[dict]) -> pd.DataFrame:
        result = pd.DataFrame(history).set_index("period").T
        result.index.name = "facility_id"
        return result

In [ ]:
# ── run ──
cfg = {
    "seed": 42,
    "n_stations": 5,
    "n_depots": 2,
    "n_periods": 2,
    "start_date": "2025-01-01",
    "trips_per_period_per_station": 8,
    "station_capacity_range": (15, 35),
    "depot_initial_stock": 50,
}

mock = BikeShareMockData(cfg).generate()
result = BikeShareSimulator().run(
    mock.inventory_initial, mock.demand, mock.returns, mock.periods,
)

In [ ]:
# Я понимаю черезе генерацию

# 1. Сначала делаю плоский подробный ноутбук, чтобы было понятно что и как работает
# 2. Потом руками рефакторю в более модульный код с классами и функциями, чтобы было удобно поддерживать и расширять.
# Я буду выстраивать этот код так, чтобы человеку это было легче всего понять.
# Для этого будет главный метод, короткий и понятный. 
# Он будет вызывать промежуточные методы.
# Они тоже будут короткими и понятными, и так далее, пока не дойдем до базовых операций с данными.
# Во время этого процесса я буду сам придумывать как и что должно быть. 
# Именно через этот процесс сформируется модель в моей голове.

# Но где тут ЭйАй а где я?

# Тоесть минималистично, есть:
# 0. В самом начале у меня есть:
# 0.1. Либо данные в файле
# 0.2. Либо данные в уже реализованной функции генерации
# 0.3. Либо данные, которые приходят из уже реализованных методов, которые я могу вызвать (входит в 0.2.)
# 1. Генерация плоской логики
# 2. Рефакторинг в классы и функции